# One-hour pump failure prediction

Run this notebook from the repository root on the Linux VM. The retained synthetic dataset is not stored in Git. Jupyter controls the verified Feast/Spark containers so their dependencies remain isolated.

Each row represents one machine at one snapshot time. Labels use `id = SHA256(machine_id|UTC timestamp)` and `label`, with a separate ID-to-entity mapping. Features use only information available at the snapshot; future labels are joined after retrieval.

The fixed 60-day protocol uses 42 training days, nine validation days and nine test days, with availability-aware purging at the boundaries. This notebook executes the same five ML stages as KFP. Holdout evaluation reproduces the existing fixed protocol; it does not select a new feature, parameter, split or threshold after observing test results.

Each fresh Feast pull stores an incremental data version. A candidate is registered only after evaluation; registration does not automatically replace the serving model.

In [ ]:
from pathlib import Path
import os, json
from ml.notebook_runtime import NotebookRuntime
from ml.protocol import load_protocol, PROTOCOL_SHA256
ROOT = Path.cwd()
protocol = load_protocol(ROOT / 'ml/protocols/medium-60d-v1.json')
runtime = NotebookRuntime(ROOT, os.environ['IOT_NOTEBOOK_RUN'], os.environ['IOT_VM_DEADLINE'])
print(json.dumps({'protocol_sha256': PROTOCOL_SHA256, 'features': protocol['feature_allowlist'],
                  'split_boundaries': protocol['chronological_split'], 'run': runtime.run}, indent=2))

## 1. Retrieve Feast features and join labels

[`ml/notebook_load.py`](../ml/notebook_load.py) exports partitions with exactly two columns, `id,label`, plus a separate entity/time mapping. Native `store.get_historical_features(entity_df=queries, features=service, filter_by_created_timestamp=False).to_df()` receives only machine/time query columns, not labels, split or episode fields.

Retrieved values are checked against immutable offline data. Two one-to-one merges attach the ID and then the label without changing row counts or label identity. Upstream feature construction establishes availability; a point-in-time join alone is insufficient. Actual build time and simulated event time are recorded separately.

In [ ]:
load_report = runtime.load()
print(json.dumps({k: load_report[k] for k in ['total_rows', 'native_rows_this_invocation', 'labels_schema', 'feast_entity_columns', 'split_counts']}, indent=2))
print(json.dumps(runtime.read('input/merge-sample.json'), indent=2))
print(json.dumps(runtime.read('lineage/automatic-data.json'), indent=2))

## 2. Split by time

Reuse `split` from [`ml/training_job.py`](../ml/training_job.py), the same kernel used by KFP. It derives partitions from event time and checks `label_available_ts < cutoff`, finite values, unique keys and class/episode support before writing three datasets. Random splitting would mix neighboring snapshots of the same failure. Test-partition metadata is not used to select a model or threshold.

In [ ]:
partitions = runtime.spark('split')
print(json.dumps(partitions['counts'], indent=2))
assert set(partitions['split_paths']) == {'train', 'validation', 'test'}

## 3. Train and select a validation threshold

The shared training kernel fits `Pipeline([VectorAssembler, StandardScaler, LogisticRegression])`. Only training data fits the scaler and inverse-frequency class weights. The fixed protocol retains `maxIter=40` and `regParam=0.1` for binary failure classification.

Validation selects a threshold from 0.1 to 0.9 by F1, then precision, then the higher threshold on a tie. Spark and portable JSON models are saved and the selection is sealed before holdout is opened. The notebook and pipeline reuse the same kernels.

In [ ]:
selection = runtime.spark('train')
print(json.dumps({'validation': selection['validation'], 'class_weights': selection['train_class_weights'],
                  'iterations': selection['spark_model_iterations'], 'scaler_fit_partition': selection['scaler_fit_partition'],
                  'holdout_rows_opened': selection['holdout_rows_opened'],
                  'saved_spark_model': selection['model_path'], 'saved_portable_model': selection['portable_path']}, indent=2))
assert selection['holdout_rows_opened'] == 0

## 4. Reload the saved model and validate

[`ml/notebook_validate.py`](../ml/notebook_validate.py) calls `PipelineModel.load`, predicts the full validation partition and checks confusion counts against the sealed selection. It also compares 200 native Spark scores with portable JSON inference. It does not change the threshold or open test data. Selection PR-AUC is Spark trapezoidal `areaUnderPR` with `numBins=0`, not average precision.

In [ ]:
verification = runtime.validate_saved_model()
print(json.dumps(verification, indent=2))
assert verification['passed'] and verification['holdout_rows_opened'] == 0
assert verification['max_abs_probability_difference'] <= 1e-12

## 5. Reproduce holdout after freezing the model and threshold

Open test data only after stage4 reloads the correct model/selection and verifies validation. Reuse the KFP evaluation kernel and the validation-selected threshold. This reproduces a previously evaluated protocol rather than introducing a newly untouched test set. Do not tune the model from this result.

In [ ]:
evaluation = runtime.spark('evaluate')
print(json.dumps({'fixed_threshold': evaluation['fixed_threshold'], 'test': evaluation['test'],
                  'portable_spark_max_abs_probability_difference': evaluation['portable_spark_max_abs_probability_difference']}, indent=2))
assert evaluation['passed'] and evaluation['fixed_threshold'] == selection['validation']['threshold']
assert evaluation['portable_spark_max_abs_probability_difference'] <= 1e-12

print(json.dumps(runtime.read('lineage/registration.json'), indent=2))

## Walkthrough and limitations

Follow arrived telemetry → features → snapshot ID → labels → temporal split → train-only fitting → validation threshold → saved-model reload → sealed holdout reproduction. The risk score is uncalibrated. The 3,132,184-row cohort comes from the separate 60-day source, not a full-100-GB training run.

This notebook uses Spark `local[2]` and sequential containers. Distributed fit is demonstrated separately by the [five-task KFP run with two executor Pods](../docs/distributed-training.md). The original notebook milestone retained its prior serving pin; later serving CI selected model6 without automatically promoting every notebook candidate.